## Quantum Circuit

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit_aer.library import SaveStatevector
import numpy as np
import matplotlib.pyplot as plt

qc = QuantumCircuit(2)




### Step 1 - Oracle
reminder:   hadimar = h()
            bit flip = x()
            controlled z = z()

In [ ]:
oracle = QuantumCircuit(2, name="oracle")
oracle.cz(0, 1)
oracle.to_gate()

oracle.draw(output='mpl', scale=.8)

## Step 2 - Difuser

In [ ]:
diffuser = QuantumCircuit(2, name="diffuser")

diffuser.h([0, 1])
diffuser.x([0, 1])

diffuser.h(1)
diffuser.cx(0, 1)
diffuser.h(1)

diffuser.x([0, 1])
diffuser.h([0, 1])

diffuser.to_gate()

diffuser.draw(output='mpl', scale=.8)

### Statevector Snapshots

In [ ]:
qc2 = QuantumCircuit(2, 2)

# init
qc2.h([0, 1])
qc2.save_statevector(label="after_H")

# oracle
qc2.compose(oracle, inplace=True)
qc2.save_statevector(label="after_oracle")

# diffuser
qc2.compose(diffuser, inplace=True)
qc2.save_statevector(label="final")

qc2.measure([0, 1], [0, 1])

### Step 3 - Run on AerSimulator

In [ ]:
sim = AerSimulator()
job = sim.run(qc2, shots=1000)
result = job.result()

counts = result.get_counts()
sv_final = result.data()['final']
sv_after_oracle = result.data()['after_oracle']
sv_after_H = result.data()['after_H']

### Graph Display

In [ ]:
probs_oracle = np.abs(sv_after_oracle)**2
probs_final = np.abs(sv_final)**2

states = ['00', '01', '10', '11']

oracle_vals = [int(probs_oracle[i] * 1000) for i in range(4)]
final_vals = [int(probs_final[i] * 1000) for i in range(4)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

bars1 = ax1.bar(states, oracle_vals, color='#FF6B6B', edgecolor='#C92A2A', linewidth=2)
ax1.set_title('After Oracle Only', fontsize=14, fontweight='bold')
ax1.set_ylabel('Measurements (out of 1000)', fontsize=12)
ax1.set_ylim(0, 1000)
ax1.grid(True, alpha=0.3, axis='y')

bars2 = ax2.bar(states, final_vals, color='#51CF66', edgecolor='#2B8A3E', linewidth=2)
ax2.set_title('After Diffuser', fontsize=14, fontweight='bold')
ax2.set_ylim(0, 1000)
ax2.grid(True, alpha=0.3, axis='y')

for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 20, 
             f'{int(height)}', ha='center', va='bottom', fontweight='bold')
    
for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 20, 
             f'{int(height)}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

